In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import category_encoders as ce

In [2]:
from fractions import Fraction

def fraction_to_decimal(fraction_str):
  """Converts a fraction string to its decimal representation."""
  try:
    return float(Fraction(fraction_str))
  except ValueError:
    return "Invalid fraction format"
  except ZeroDivisionError:
      return "Cannot divide by zero"

In [3]:
df = pd.read_csv("../output/Bach_chordify_int_data.csv")

display(df.head())
print(df['duration'].dtype)
print(df['duration'].head().map(type))
df['duration'] = df['duration'].apply(Fraction)
df['duration'] = df['duration'].astype(float)
df.drop(['file', 'position'], axis=1, inplace=True)
display(df.head())
X = df.drop(columns=['label'])
y = df['label']
print("Training samples:", X.shape[0])
print("Feature dims:",    X.shape[1])
display(X.head())
display(y.head())

,lookback_1,label,duration,file,position
0,0,1,1/3,"Bach, Carl Philipp Emanuel, Solfeggio in A maj...",0
1,1,129,1/6,"Bach, Carl Philipp Emanuel, Solfeggio in A maj...",1
2,129,128,1/6,"Bach, Carl Philipp Emanuel, Solfeggio in A maj...",2
3,128,1,0.25,"Bach, Carl Philipp Emanuel, Solfeggio in A maj...",3
4,1,6144,1/12,"Bach, Carl Philipp Emanuel, Solfeggio in A maj...",4


object
0    <class 'str'>
1    <class 'str'>
2    <class 'str'>
3    <class 'str'>
4    <class 'str'>
Name: duration, dtype: object


,lookback_1,label,duration
0,0,1,0.333333
1,1,129,0.166667
2,129,128,0.166667
3,128,1,0.250000
4,1,6144,0.083333


Training samples: 200525
Feature dims: 2


,lookback_1,duration
0,0,0.333333
1,1,0.166667
2,129,0.166667
3,128,0.250000
4,1,0.083333


0       1
1     129
2     128
3       1
4    6144
Name: label, dtype: int64

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
import os
import sys

def has_nvidia_gpu():
    try:
        import pynvml
        pynvml.nvmlInit()
        count = pynvml.nvmlDeviceGetCount()
        pynvml.nvmlShutdown()
        return count > 0
    except Exception:
        return False

def has_intel_cpu():
    try:
        import cpuinfo
        info = cpuinfo.get_cpu_info()
        return info.get('vendor_id_raw', '').lower() == 'genuineintel'
    except ImportError:
        # If cpuinfo isn’t installed, you can also parse /proc/cpuinfo on Linux:
        if sys.platform.startswith('linux'):
            with open('/proc/cpuinfo') as f:
                return any('intel' in line.lower() for line in f if line.startswith('vendor_id'))
        return False

if has_nvidia_gpu():
    print("→ NVIDIA GPU detected. Patching to use cuML SVC.")
    from cuml.svm import SVC as cuSVC
    from cuml.model_selection import GridSearchCV as cuGrid
    SVC_BACKEND = 'gpu'
    
elif has_intel_cpu():
    print("→ Intel CPU detected. Patching scikit-learn with sklearnex.")
    try:
        from sklearnex import patch_sklearn
        patch_sklearn()  
    except ImportError:
        print("   ⚠️ sklearnex not installed; falling back to vanilla scikit-learn.")
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    SVC_BACKEND = 'intel'

else:
    print("→ No GPU or Intel‐patched CPU found; using vanilla scikit-learn.")
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    SVC_BACKEND = 'cpu'


→ Intel CPU detected. Patching scikit-learn with sklearnex.


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [ ]:
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

pipe = make_pipeline(
    StandardScaler(),
    SVC(random_state=42)
)

param_grid = {
    'svc__C':    [0.1, 1, 10],
    'svc__gamma':['auto'],
    'svc__kernel':['rbf','linear']
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

GridClass = cuGrid if SVC_BACKEND == 'gpu' else GridSearchCV

grid = GridClass(
    estimator=pipe,
    param_grid=param_grid,
    cv=kf,
    n_jobs=-1,        # <-- must use joblib parallelism
    verbose=0,        # turn off sklearn’s text spew
    scoring='accuracy'
)

# total = n_params * n_splits
n_candidates = len(param_grid['svc__C']) * \
               len(param_grid['svc__gamma']) * \
               len(param_grid['svc__kernel'])
total_iters = n_candidates * kf.get_n_splits()

with tqdm_joblib(tqdm(desc="GridSearchCV", total=total_iters)) as progress:
    grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print(f"Mean CV accuracy: {grid.best_score_:.3f}")
y_pred = grid.predict(X_test)


GridSearchCV:   0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
clf = make_pipeline(StandardScaler(), LinearSVC(max_iter=5000, random_state=42))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [ ]:

np.savetxt('predictions.txt', y_pred, fmt='%d')
"""
print("\nTest set performance:")
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
"""